# Simple Test Fixture Generation

This notebook creates test fixtures using direct FFmpeg commands, following the same pattern as CreateTranscriptProd.ipynb.

**Key Principles:**
- No imports from minutia modules (avoids circular dependency)
- Direct FFmpeg usage for reliability
- Step-by-step process with manual verification
- Creates small, fast test clips (5-15 seconds)

**Generated Fixtures:**
- `basic_5s.mkv` - Clear speech, good audio quality
- `noisy_10s.mkv` - Background noise, harder transcription
- `quiet_8s.mkv` - Very quiet audio, edge case testing

**Output Directory:** `tests/fixtures/`

## Setup

In [ ]:
import subprocess
import json
from pathlib import Path
import shutil

# Paths
recordings_dir = Path('../../+recordings')
fixtures_dir = Path('../fixtures')
video_fixtures = fixtures_dir / 'video'
audio_fixtures = fixtures_dir / 'audio'

# Create directories
video_fixtures.mkdir(parents=True, exist_ok=True)
audio_fixtures.mkdir(parents=True, exist_ok=True)

print(f"Recordings source: {recordings_dir}")
print(f"Video fixtures: {video_fixtures}")
print(f"Audio fixtures: {audio_fixtures}")

## Available Source Recordings

In [ ]:
# List available recordings
mkv_files = list(recordings_dir.glob('*.mkv'))
print(f"Found {len(mkv_files)} MKV recordings:")
for mkv in mkv_files:
    print(f"  - {mkv.name}")
    
    # Get duration using ffprobe
    cmd = ['ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_format', str(mkv)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        data = json.loads(result.stdout)
        duration = float(data['format']['duration'])
        print(f"    Duration: {duration:.1f}s")
    else:
        print(f"    Could not get duration")

## Fixture 1: Basic Clear Speech (5 seconds)

Extract a 5-second clip with clear speech from Example-Microphone.mkv

In [ ]:
# Basic 5-second fixture
source_file = recordings_dir / 'Example-Microphone.mkv'
output_file = video_fixtures / 'basic.mkv'

cmd = [
    'ffmpeg', '-y',  # -y to overwrite
    '-i', str(source_file),
    '-c', 'copy',  # Copy without re-encoding
    str(output_file)
]

print(f"Creating basic.mkv from {source_file.name}...")
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    print(f"✓ Created {output_file}")
    
    # Verify the clip
    cmd_verify = ['ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_format', '-show_streams', str(output_file)]
    verify_result = subprocess.run(cmd_verify, capture_output=True, text=True)
    if verify_result.returncode == 0:
        data = json.loads(verify_result.stdout)
        duration = float(data['format']['duration'])
        audio_streams = [s for s in data['streams'] if s['codec_type'] == 'audio']
        print(f"  Duration: {duration:.1f}s")
        print(f"  Audio tracks: {len(audio_streams)}")
else:
    print(f"✗ Error creating fixture: {result.stderr}")

## Fixture 2: Noisy Audio (10 seconds)

Extract a 10-second clip with background noise from Example-Noisy.mkv

In [ ]:
# Noisy 10-second fixture
source_file = recordings_dir / 'Example-Noisy.mkv'
output_file = video_fixtures / 'noisy.mkv'

# Check if source exists
if not source_file.exists():
    print(f"⚠ Source file {source_file.name} not found, skipping noisy fixture")
else:
    # Extract 10 seconds starting from 2 seconds
    cmd = [
        'ffmpeg', '-y',
        '-i', str(source_file),
        '-c', 'copy',
        str(output_file)
    ]
    
    print(f"Creating noisy.mkv from {source_file.name}...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f"✓ Created {output_file}")
        
        # Verify
        cmd_verify = ['ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_format', '-show_streams', str(output_file)]
        verify_result = subprocess.run(cmd_verify, capture_output=True, text=True)
        if verify_result.returncode == 0:
            data = json.loads(verify_result.stdout)
            duration = float(data['format']['duration'])
            audio_streams = [s for s in data['streams'] if s['codec_type'] == 'audio']
            print(f"  Duration: {duration:.1f}s")
            print(f"  Audio tracks: {len(audio_streams)}")
    else:
        print(f"✗ Error creating fixture: {result.stderr}")

## Fixture 3: Complex Audio (8 seconds)

Extract an 8-second clip from Example-Song+MLKSpeech.mkv (if available)

In [ ]:
# Complex audio fixture (song + speech)
source_file = recordings_dir / 'Example-Song+MLKSpeech.mkv'
output_file = video_fixtures / 'complex.mkv'

if not source_file.exists():
    print(f"⚠ Source file {source_file.name} not found, skipping complex fixture")
else:
    # Extract 8 seconds starting from 3 seconds
    cmd = [
        'ffmpeg', '-y',
        '-i', str(source_file),
        '-c', 'copy',
        str(output_file)
    ]
    
    print(f"Creating complex.mkv from {source_file.name}...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f"✓ Created {output_file}")
        
        # Verify
        cmd_verify = ['ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_format', '-show_streams', str(output_file)]
        verify_result = subprocess.run(cmd_verify, capture_output=True, text=True)
        if verify_result.returncode == 0:
            data = json.loads(verify_result.stdout)
            duration = float(data['format']['duration'])
            audio_streams = [s for s in data['streams'] if s['codec_type'] == 'audio']
            print(f"  Duration: {duration:.1f}s")
            print(f"  Audio tracks: {len(audio_streams)}")
    else:
        print(f"✗ Error creating fixture: {result.stderr}")

## Extract Audio Tracks

Extract audio tracks from each video fixture for direct audio testing

In [ ]:
# Extract audio tracks from each video fixture
video_fixtures_list = list(video_fixtures.glob('*.mkv'))

for video_file in video_fixtures_list:
    print(f"\nExtracting audio from {video_file.name}...")
    
    # Get audio stream info first
    cmd_probe = ['ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_streams', str(video_file)]
    probe_result = subprocess.run(cmd_probe, capture_output=True, text=True)
    
    if probe_result.returncode == 0:
        data = json.loads(probe_result.stdout)
        audio_streams = [s for s in data['streams'] if s['codec_type'] == 'audio']
        
        # Extract each audio track
        for i, stream in enumerate(audio_streams):
            track_num = i + 1  # 1-indexed track numbers
            output_name = f"{video_file.stem}_track_{track_num}.wav"
            output_path = audio_fixtures / output_name
            
            cmd_extract = [
                'ffmpeg', '-y',
                '-i', str(video_file),
                '-map', f'0:a:{i}',  # Select specific audio track
                '-acodec', 'pcm_s16le',  # Convert to WAV
                str(output_path)
            ]
            
            extract_result = subprocess.run(cmd_extract, capture_output=True, text=True)
            
            if extract_result.returncode == 0:
                print(f"  ✓ Track {track_num}: {output_name}")
            else:
                print(f"  ✗ Failed to extract track {track_num}: {extract_result.stderr[:100]}")
    else:
        print(f"  ✗ Could not probe {video_file.name}")

## Summary

List all created fixtures

In [ ]:
print("=== Created Test Fixtures ===")
print("\nVideo fixtures:")
video_files = sorted(video_fixtures.glob('*.mkv'))
for vf in video_files:
    size_mb = vf.stat().st_size / (1024 * 1024)
    print(f"  {vf.name} ({size_mb:.1f} MB)")

print("\nAudio fixtures:")
audio_files = sorted(audio_fixtures.glob('*.wav'))
for af in audio_files:
    size_mb = af.stat().st_size / (1024 * 1024)
    print(f"  {af.name} ({size_mb:.1f} MB)")

print(f"\nTotal: {len(video_files)} video clips, {len(audio_files)} audio tracks")

# Save a simple metadata file
metadata = {
    "generated_by": "tests/scripts/generate_test_fixtures_simple.ipynb",
    "description": "Simple test fixtures created with direct FFmpeg commands",
    "video_fixtures": [vf.name for vf in video_files],
    "audio_fixtures": [af.name for af in audio_files],
    "notes": [
        "These are minimal test fixtures for fast unit/integration testing",
        "Video clips are 5-10 seconds to keep test suite fast",
        "Audio tracks extracted separately for direct audio testing"
    ]
}

metadata_file = fixtures_dir / 'simple_fixtures_metadata.json'
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\nMetadata saved to: {metadata_file}")

## Next Steps

**To create ground truth data:**
1. Run the working pipeline (CreateTranscriptProd.ipynb approach) on each fixture
2. Save the outputs as "golden samples" for regression testing
3. Document expected behavior for each test case

**Testing approach:**
- Use these fixtures in unit tests (with mocked dependencies)
- Run integration tests with real Whisper tiny model
- Compare outputs against golden samples for regression detection